# **🚘 LTO-ASSIST: Driver's License Application & Renewal Q&A System Using LLM with Prompt Engineering (Notebook)**

**NLP1000 Machine Project - Prompt Engineering Version**

**Group Members:**
- BALINGIT, Andrei Luis
- BURAYAG, Ethan Axl

**Section:** 
- NLP1000 - SO1

**Notebook Overview:** 
- This notebook walks through the development of LTO-ASSIST, an LLM-powered Q&A system that helps Filipino motorists with driver's license application and renewal queries. We will iterate through three versions of our system prompt, observe what each version gets wrong, and document our refinements.

---

## **Section 1: Setup and Installation**

Before anything else, we need to install the required libraries and load our API key.

**Libraries we need:**
- `google-genai` — the official Gemini SDK
- `python-dotenv` — to safely load our API key from a `.env` file

Run the cell below once to install them.

In [1]:
# Install required libraries
# Only needs to be run once
%pip install google-genai python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


**Loading the API Key**

Make sure you have a `.env` file in the same folder as this notebook with the following contents:

```
GEMINI_API_KEY=your_actual_api_key_here
```

We load it using `dotenv` so the key is never hardcoded in our code.

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Load API key from .env file
load_dotenv()

# Set up Gemini client
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("✅ Setup complete. Gemini client is ready.")

✅ Setup complete. Gemini client is ready.


## **Section 2: Helper Function**

Instead of rewriting the Gemini API call every time, we define one reusable function called `ask()`. It takes two inputs:
- `system_prompt` — the instructions we give the LLM about how to behave
- `query` — the user's question

It prints the response neatly so we can observe and compare outputs across prompt versions.


In [3]:
def ask(system_prompt, query):
    """
    Sends a query to Gemini with a given system prompt.
    Prints the response to the screen.
    """
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        config=types.GenerateContentConfig(
            system_instruction=system_prompt
        ),
        contents=query
    )

    print(f"🧑 USER: {query}")
    print(f"\n🤖 LTO-ASSIST:\n{response.text}")
    print("-" * 60)

## **Section 3: Test Queries**

These are the four standard queries we will use to test each version of our system prompt. They cover our defined scope and include one Taglish query and one edge case.

| # | Type | Query |
|---|---|---|
| Q1 | New Application | What are the requirements to apply for a new non-professional driver's license? |
| Q2 | Renewal Procedure | How do I renew my non-professional license? Do I need to take an exam? |
| Q3 | Fee Inquiry (Taglish) | Magkano ang bayad para sa renewal ng non-professional license ko? |
| Q4 | Edge Case (Sarcastic) | So I just need to bribe someone diba? Mas mabilis pa siguro yun. |

In [4]:
# Our four standard test queries
test_queries = [
    "What are the requirements to apply for a new non-professional driver's license?",
    "How do I renew my non-professional license? Do I need to take an exam?",
    "Magkano ang bayad para sa renewal ng non-professional license ko?",
    "So I just need to bribe someone diba? Mas mabilis pa siguro yun."
]

## **Section 4: Version 1 - Initial Prompt**

**What we tried**

A minimal, generic prompt. We gave the LLM a basic role.

**What we expected**

A focused assistant that answers LTO questions correctly.

**What we observed *(to be filled after running)***
- [✅] Did it stay on topic?
- [❌] Was the format consistent?
- [❌] Did it not hallucinate fees or steps?
- [❌] Did it handle Tagalog/Taglish correctly?
- [✅] Did it handle the sarcastic query well?

In [5]:
# VERSION 1 — Initial System Prompt
system_prompt_v1 = (
    "You are a helpful assistant for the Philippine Land Transportation Office (LTO)." 
    "Answer the user's question about driver's license applications and renewals based on the provided context."
    "\nCURRENT TASK:\n"
    "Context: {context}\n"
    "User Query: {query}"
)

In [6]:
# Run all test queries against Version 1
print("=" * 60)
print("SYSTEM PROMPT VERSION 1 RESULTS")
print("=" * 60)

for query in test_queries:
    ask(system_prompt_v1, query)

SYSTEM PROMPT VERSION 1 RESULTS
🧑 USER: What are the requirements to apply for a new non-professional driver's license?

🤖 LTO-ASSIST:
To apply for a new non-professional driver's license, you need to prepare the following:

*   **Philippine Statistics Authority (PSA) Authenticated Birth Certificate:** This is a crucial document to prove your identity and age.
*   **Valid Government-Issued ID:** Bring any valid ID with your picture and signature, such as a passport, PhilHealth ID, SSS ID, etc.
*   **Drug Test Certificate:** You must undergo and pass a drug test from a DOH-accredited drug testing facility.
*   **Medical Certificate:** This certificate must be issued by a DOH-accredited physician, indicating that you are physically and mentally fit to drive.
*   **Proof of Application (if applicable):** If you have already completed an application online, bring the necessary proof.

**Additional requirements for applicants aged 18-23:**

*   **Parental Consent:** If you are between 18 an

**V1 Observations**

**What went wrong:**
- **Hallucination & Prior Knowledge Bleed:** Because the prompt only said "based on the provided context" without strict constraints, the LLM sometimes supplemented incomplete or wrong information with its own outdated pre-training data (e.g. incomplete requirements).
- **Formatting Inconsistency:** It returned plain text, sometimes using bullet points, sometimes long paragraphs, making it impossible to parse systematically.
- **Language Mismatch:** When asked a question in Taglish, it sometimes responded in pure, formal English, breaking the conversational goal of the system.
- **Non-Reliability:** When asked a question, it still asks the user to refer to the website for information confirmation, breaking the reliability goal of the system.

**What needs to change for V2:**
- Add a strict grounding rule to prevent the LLM from supplementing retrieved context with outdated pre-training data, which makes the system untrustworthy for users acting on fee or requirement information.
- Enforce a structured JSON output format to ensure responses are consistent and programmatically parseable, especially for the ablation study in Phase 4.
- Add an explicit language matching rule to ensure Tagalog and Taglish queries receive responses in kind, since formal English responses break the accessibility goal of the system.
- Add a reliability rule that forces the LLM to answer directly from context instead of deflecting to external websites, which defeats the entire purpose of building the assistant.